#  **KOFIC 1~4주차 주말 관객수 수집**
##->수동 수집 영화 목록을 확인 후, 수동 수집 이후 'kofic_최종_영화 드롭률 데이터셋.xlsx'파일 생성

##  본 노트북의 목적

KOFIC 오픈 API를 통해 2023~2025년 개봉한 한국·외국 영화의

**1~4주차 주말(금+토+일) 관객수**를 수집합니다. 이 데이터는 본 프로젝트의 **타겟 변수(드롭률)** 와

**흥행 지속성 측정** 의 근거가 됩니다.

##  사용 API

- **KOFIC 오픈 API** (영화진흥위원회)
- API 키 발급: https://www.kobis.or.kr/kobisopenapi
- 무료, 일일 호출 제한 충분

##  수집 설계

### 체급 분류 기준 (누적관객수)
| 체급 | 기준 | 표본 수 |
|---|---|---|
| 중형 | 50만 ~ 300만 명 | 약 102편 |
| 중대형 | 300만 ~ 500만 명 | 약 14편 |
| 대형 | 500만 명 이상 | 약 14편 |
| **합계** | — | **약 130편** |

### N주차 주말관객 정의
```
1주차 = 개봉 후 첫 금/토/일 관객수 합
2주차 = 1주차 + 7일 후 금/토/일 관객수 합
3주차 = 2주차 + 7일 후 금/토/일 관객수 합
4주차 = 3주차 + 7일 후 금/토/일 관객수 합
   ↑ 각 주차 독립적, 누적 아님
```



### 왜 금+토+일인가?
KOFIC 공식 *"주말(weekGb=1)"* 정의에 따라 금토일을 주말로 보았습니다. 평일(월~목)은 영화관 방문 패턴이 주말과 크게 달라 노이즈가 커지므로 제외했습니다.

##  출력

`kofic_최종_영화_드롭률_데이터셋.xlsx` — 약 387행 (130편 × 3행 패널 구조)
- 영화명, 개봉일, 누적관객수, 체급
- 기준주차 (1, 2, 3)
- 현재주_주말관객, 다음주_주말관객
- 드롭률(%) = (현재주 − 다음주) / 현재주 × 100

##  주의사항

- KOFIC API는 박스오피스 TOP 50만 반환 → 3·4주차 결측 영화 일부 발생
- 결측 영화는 KOFIC 통합전산망에서 **수동 보완**

## **1️. 라이브러리 임포트**

###  작업 내용
KOFIC API 호출과 데이터 처리에 필요한 라이브러리를 불러옵니다.

###  사용 라이브러리

| 라이브러리 | 용도 |
|---|---|
| `requests` | KOFIC REST API 호출 |
| `pandas` | 데이터프레임 처리 + 엑셀 입출력 |
| `datetime`, `timedelta` | 개봉일 기준 N주차 날짜 계산 |
| `time` | API 부하 방지용 sleep |
| `tqdm` | 130편 수집 진행률 표시 |
| `google.colab.files` | 코랩 파일 업로드·다운로드 |

코랩에는 `requests`, `pandas`가 기본 설치되어 있어 별도 `!pip install`이 필요 없습니다.

In [1]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta
from tqdm import tqdm

# ─────────────────────────────────────────
#  설정 (여기만 수정)
# ─────────────────────────────────────────
API_KEY      = "66dd97097c3b50d89e66316bfd084737"
START_YEAR   = 2023
END_YEAR     = 2025
BASE_URL     = "http://www.kobis.or.kr/kobisopenapi/webservice/rest"

# 체급 기준 (누적관객수)
TIER_RULES = {
    "중형":   (500_000,   3_000_000),
    "중대형": (3_000_000, 5_000_000),
    "대형":   (5_000_000, float("inf")),
}

# ─────────────────────────────────────────
# STEP 1. 연도별 주말 박스오피스 전체 수집
#         → 누적관객수 기준으로 체급 분류
# ─────────────────────────────────────────
def get_weekly_boxoffice(target_dt: str, week_gb: str = "1") -> list:
    """
    주간/주말 박스오피스 TOP 50 반환
    week_gb: "0"=주간, "1"=주말(금토일), "2"=주중
    target_dt: 해당 주 월요일 YYYYMMDD
    """
    url = f"{BASE_URL}/boxoffice/searchWeeklyBoxOfficeList.json"
    params = {
        "key":      API_KEY,
        "targetDt": target_dt,
        "weekGb":   week_gb,
    }
    try:
        resp = requests.get(url, params=params, timeout=15)
        data = resp.json()
        return data.get("boxOfficeResult", {}).get("weeklyBoxOfficeList", [])
    except Exception as e:
        print(f"  [ERROR] {target_dt}: {e}")
        return []


def get_all_mondays(year: int) -> list:
    """해당 연도의 모든 월요일 반환"""
    mondays = []
    dt = datetime(year, 1, 1)
    # 첫 번째 월요일로 이동
    while dt.weekday() != 0:
        dt += timedelta(days=1)
    while dt.year == year:
        mondays.append(dt.strftime("%Y%m%d"))
        dt += timedelta(weeks=1)
    return mondays


def collect_all_movies_by_year(start_year: int, end_year: int) -> pd.DataFrame:
    """
    연도별 주말 박스오피스를 전체 수집해서
    영화별 누적관객수·개봉일 파악
    """
    movie_dict = {}  # {영화명: {개봉일, 누적관객수, ...}}

    for year in range(start_year, end_year + 1):
        mondays = get_all_mondays(year)
        print(f"\n[{year}년] 총 {len(mondays)}주차 수집 중...")

        for monday in tqdm(mondays, desc=f"{year}"):
            items = get_weekly_boxoffice(monday, week_gb="1")
            for item in items:
                movie_nm   = item.get("movieNm", "")
                open_dt    = item.get("openDt", "")
                rank       = int(item.get("rank", 99))
                audi_acc   = int(item.get("audiAcc", 0))   # 누적관객수
                audi_cnt   = int(item.get("audiCnt", 0))   # 해당주 관객수

                if not movie_nm or not open_dt:
                    continue

                # 누적관객수 최대값으로 업데이트 (가장 최신 기록)
                if movie_nm not in movie_dict:
                    movie_dict[movie_nm] = {
                        "영화명":    movie_nm,
                        "개봉일":    open_dt,
                        "누적관객수": audi_acc,
                    }
                else:
                    if audi_acc > movie_dict[movie_nm]["누적관객수"]:
                        movie_dict[movie_nm]["누적관객수"] = audi_acc
                        movie_dict[movie_nm]["개봉일"]    = open_dt

            time.sleep(0.2)

    movie_df = pd.DataFrame(list(movie_dict.values()))
    print(f"\n총 수집 영화: {len(movie_df)}편")
    return movie_df


# ─────────────────────────────────────────
# STEP 2. 체급 분류
# ─────────────────────────────────────────
def classify_tier(audi_acc: int) -> str:
    for tier, (lo, hi) in TIER_RULES.items():
        if lo < audi_acc < hi:
            return tier
    return None  # 범위 밖 (50만 미만 등)


# ─────────────────────────────────────────
# STEP 3. 개봉연도 필터링
# ─────────────────────────────────────────
def filter_by_open_year(movie_df: pd.DataFrame,
                        start_year: int, end_year: int) -> pd.DataFrame:
    def parse_year(dt_str):
        try:
            return int(str(dt_str)[:4])
        except:
            return 0

    movie_df["개봉연도"] = movie_df["개봉일"].apply(parse_year)
    filtered = movie_df[
        (movie_df["개봉연도"] >= start_year) &
        (movie_df["개봉연도"] <= end_year)
    ].copy()
    return filtered


# ─────────────────────────────────────────
# STEP 4. 주차별 주말 관객수 수집
# ─────────────────────────────────────────
def get_week_monday(open_date_str: str, week: int) -> str:
    """개봉일 기준 N주차의 월요일(조회기준일) 반환"""
    open_dt = datetime.strptime(str(open_date_str)[:8], "%Y%m%d")
    days_since_monday = open_dt.weekday()  # 0=월
    first_monday = open_dt - timedelta(days=days_since_monday)
    target_monday = first_monday + timedelta(weeks=week - 1)
    return target_monday.strftime("%Y%m%d")


def get_weekly_audience_for_movie(movie_name: str,
                                   open_date: str,
                                   week: int) -> int:
    """
    특정 영화의 N주차 주말(금+토+일) 관객수 반환
    주간 박스오피스 TOP50에 없으면 0 반환
    """
    monday = get_week_monday(open_date, week)
    items  = get_weekly_boxoffice(monday, week_gb="1")

    for item in items:
        if movie_name in item.get("movieNm", ""):
            return int(item.get("audiCnt", 0))
    return 0  # TOP50 밖


def collect_weekly_audience(movie_df: pd.DataFrame) -> pd.DataFrame:
    """전체 영화에 대해 1~4주차 주말 관객수 수집"""
    results = []

    for _, row in tqdm(movie_df.iterrows(),
                       total=len(movie_df),
                       desc="주차별 관객수 수집"):
        movie_name = row["영화명"]
        open_date  = str(row["개봉일"]).replace('-', '') # Modified this line

        result = {
            "영화명":    movie_name,
            "개봉일":    open_date,
            "개봉연도":  row["개봉연도"],
            "누적관객수": row["누적관객수"],
            "체급":      row["체급"],
        }

        for week in [1, 2, 3, 4]:
            aud = get_weekly_audience_for_movie(movie_name, open_date, week)
            result[f"{week}주차_주말관객"] = aud
            time.sleep(0.25)

        results.append(result)
        time.sleep(0.3)

    return pd.DataFrame(results)


# ─────────────────────────────────────────
# STEP 5. 드롭률 계산 + Long Format 변환
# ─────────────────────────────────────────
def calculate_droprate_long(df: pd.DataFrame) -> pd.DataFrame:
    """주차별 드롭률 계산 + Long Format 변환"""
    rows = []

    for _, r in df.iterrows():
        weeks = {
            1: r["1주차_주말관객"],
            2: r["2주차_주말관객"],
            3: r["3주차_주말관객"],
            4: r["4주차_주말관객"],
        }

        base = {
            "영화명":    r["영화명"],
            "개봉일":    r["개봉일"],
            "개봉연도":  r["개봉연도"],
            "누적관객수": r["누적관객수"],
            "체급":      r["체급"],
            "연도_2024": int(r["개봉연도"] == 2024),
            "연도_2025": int(r["개봉연도"] == 2025),
        }

        for base_week in [1, 2, 3]:
            curr = weeks.get(base_week, 0)
            nxt  = weeks.get(base_week + 1, 0)

            # 데이터 없으면 스킵
            if curr == 0 or nxt == 0:
                continue

            drop_rate = (curr - nxt) / curr * 100

            # 극단 이상치 제거
            if drop_rate < -200 or drop_rate > 100:
                continue

            rows.append({
                **base,
                "기준주차":           base_week,
                "현재주_주말관객":     curr,
                "다음주_주말관객":     nxt,
                "주차_드롭률(%)":      round(drop_rate, 4),
                "log_현재주_관객":     np.log1p(curr),
                "log_누적관객수":      np.log1p(r["누적관객수"]),
                "주차_2차":            int(base_week == 2),
                "주차_3차":            int(base_week == 3),
            })

    return pd.DataFrame(rows)


# ─────────────────────────────────────────
# MAIN 실행
# ─────────────────────────────────────────
if __name__ == "__main__":

    print("=" * 60)
    print("KOFIC 주차별 주말 관객수 수집기")
    print(f"대상: {START_YEAR}~{END_YEAR}년 개봉작")
    print(f"체급 기준: 중형(50만~300만) / 중대형(300만~500만) / 대형(500만+)")
    print("=" * 60)

    # STEP 1. 전체 영화 목록 수집
    print("\n[STEP 1] 박스오피스에서 전체 영화 목록 수집...")
    all_movies = collect_all_movies_by_year(START_YEAR, END_YEAR)

    # STEP 2. 체급 분류
    print("\n[STEP 2] 체급 분류...")
    all_movies["체급"] = all_movies["누적관객수"].apply(classify_tier)
    target_movies = all_movies[all_movies["체급"].notna()].copy()

    # STEP 3. 개봉연도 필터링
    print("\n[STEP 3] 개봉연도 필터링...")
    target_movies = filter_by_open_year(target_movies, START_YEAR, END_YEAR)

    # 중간 저장
    target_movies.to_excel("movie_list_classified.xlsx", index=False)
    print(f"\n체급별 영화 수:")
    print(target_movies["체급"].value_counts())
    print(f"총 대상: {len(target_movies)}편")

    # STEP 4. 주차별 주말 관객수 수집
    print("\n[STEP 4] 1~4주차 주말 관객수 수집...")
    weekly_raw = collect_weekly_audience(target_movies)
    weekly_raw.to_excel("kofic_weekly_raw.xlsx", index=False)
    print(f"\n 원본 저장: kofic_weekly_raw.xlsx")

    # 0인 행 확인
    for w in [1, 2, 3, 4]:
        col = f"{w}주차_주말관객"
        zero_n = (weekly_raw[col] == 0).sum()
        print(f"  {w}주차 = 0 (수동 확인 필요): {zero_n}편")

    # STEP 5. 드롭률 + Long Format
    print("\n[STEP 5] 드롭률 계산 + Long Format 변환...")
    long_df = calculate_droprate_long(weekly_raw)
    long_df.to_excel("kofic_long_format.xlsx", index=False)

    # 최종 요약
    print("\n" + "=" * 60)
    print(" 수집 완료!")
    print(f"  원본:       kofic_weekly_raw.xlsx    ({len(weekly_raw)}편)")
    print(f"  Long Format: kofic_long_format.xlsx  ({len(long_df)}행)")
    print(f"\n체급별 행 수:")
    print(long_df["체급"].value_counts())
    print(f"\n연도별 행 수:")
    print(long_df["개봉연도"].value_counts().sort_index())
    print(f"\n드롭률 기술통계:")
    print(long_df["주차_드롭률(%)"].describe().round(2))

    # 0인 행 목록 별도 저장 (수동 수집용)
    zero_rows = weekly_raw[
        (weekly_raw["3주차_주말관객"] == 0) |
        (weekly_raw["4주차_주말관객"] == 0)
    ][["영화명", "개봉일", "개봉연도", "체급",
       "1주차_주말관객", "2주차_주말관객",
       "3주차_주말관객", "4주차_주말관객"]].copy()

    zero_rows["3주차_수동입력"] = ""
    zero_rows["4주차_수동입력"] = ""
    zero_rows.to_excel("수동수집_필요목록.xlsx", index=False)
    print(f"\n⚠ 수동 수집 필요: {len(zero_rows)}편 → 수동수집_필요목록.xlsx")

KOFIC 주차별 주말 관객수 수집기
대상: 2023~2025년 개봉작
체급 기준: 중형(50만~300만) / 중대형(300만~500만) / 대형(500만+)

[STEP 1] 박스오피스에서 전체 영화 목록 수집...

[2023년] 총 52주차 수집 중...


2023: 100%|██████████| 52/52 [03:54<00:00,  4.51s/it]



[2024년] 총 53주차 수집 중...


2024: 100%|██████████| 53/53 [03:39<00:00,  4.15s/it]



[2025년] 총 52주차 수집 중...


2025: 100%|██████████| 52/52 [02:40<00:00,  3.08s/it]



총 수집 영화: 536편

[STEP 2] 체급 분류...

[STEP 3] 개봉연도 필터링...

체급별 영화 수:
체급
중형     100
중대형     14
대형      14
Name: count, dtype: int64
총 대상: 128편

[STEP 4] 1~4주차 주말 관객수 수집...


주차별 관객수 수집: 100%|██████████| 128/128 [15:36<00:00,  7.32s/it]


 원본 저장: kofic_weekly_raw.xlsx
  1주차 = 0 (수동 확인 필요): 0편
  2주차 = 0 (수동 확인 필요): 0편
  3주차 = 0 (수동 확인 필요): 3편
  4주차 = 0 (수동 확인 필요): 19편

[STEP 5] 드롭률 계산 + Long Format 변환...

 수집 완료!
  원본:       kofic_weekly_raw.xlsx    (128편)
  Long Format: kofic_long_format.xlsx  (361행)

체급별 행 수:
체급
중형     277
중대형     42
대형      42
Name: count, dtype: int64

연도별 행 수:
개봉연도
2023    121
2024    134
2025    106
Name: count, dtype: int64

드롭률 기술통계:
count    361.00
mean      43.44
std       26.18
min     -120.67
25%       31.90
50%       46.53
75%       60.32
max       90.71
Name: 주차_드롭률(%), dtype: float64

⚠ 수동 수집 필요: 19편 → 수동수집_필요목록.xlsx
